# Resolve a CSV batch

Required CSV columns: `main_text`, `country_code`  
Optional columns: `ean`, `retailer_name`, `row_id`

Rows are processed sequentially for simple debugging. Output is saved after every row.

In [ ]:
import os, sys
from pathlib import Path
from dotenv import load_dotenv

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
os.chdir(ROOT)
load_dotenv(ROOT / ".env")
sys.path.insert(0, str(ROOT / "src"))

from product_url_finder import ProductInput, Settings, resolve_product

settings = Settings.from_env()
settings.validate()
print("Ready")
print(f"SERP budget: {settings.serp_calls}")
print(f"Crawl budget: {settings.crawl_candidates}")
print(f"PCA LLM enabled: {settings.llm_enabled}")


In [ ]:
import pandas as pd

INPUT_CSV = ROOT / "samples" / "products.csv"
OUTPUT_CSV = ROOT / "data" / "product_urls.csv"

source = pd.read_csv(INPUT_CSV, dtype=str, keep_default_na=False)
required = {"main_text", "country_code"}
missing = required - set(source.columns)
if missing:
    raise ValueError(f"Missing mandatory columns: {sorted(missing)}")

for column in ("ean", "retailer_name", "row_id"):
    if column not in source.columns:
        source[column] = ""

products = []
for index, row in source.iterrows():
    products.append(ProductInput(
        main_text=row["main_text"],
        country_code=row["country_code"],
        ean=row["ean"] or None,
        retailer_name=row["retailer_name"] or None,
        row_id=row["row_id"] or f"ROW-{index + 1:06d}",
    ).normalized())

row_ids = [item.row_id for item in products]
if len(row_ids) != len(set(row_ids)):
    raise ValueError("row_id values must be unique.")

print(f"Validated {len(products)} rows before any paid search call.")


In [ ]:
rows = []
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

for index, product in enumerate(products, start=1):
    print(f"[{index}/{len(products)}] {product.row_id}: {product.main_text[:80]}")
    result = await resolve_product(product, settings)
    rows.append({
        "ROW_ID": result["row_id"],
        "MAIN_TEXT": result["main_text"],
        "COUNTRY": result["country_code"],
        "RETAILER": result["retailer_name"],
        "EAN": result["ean"],
        "CANDIDATE_URLS": " | ".join(result["candidate_urls"]),
        "PRODUCT_URL": result["product_url"],
        "CONFIDENCE": result["confidence"],
        "VALIDATION_STATUS": result["status"],
        "IDENTITY_STATUS": result["identity_status"],
        "RETAILER_CHECK": result["retailer_check"],
        "JUSTIFICATION": result["justification"],
        "ARTIFACT_DIR": result["artifact_dir"],
    })
    pd.DataFrame(rows).to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

output = pd.DataFrame(rows)
print(f"Saved: {OUTPUT_CSV}")
output
